In [ ]:
%load_ext autoreload
%autoreload 2
from thesis.metrics import *

style_dict = {
        # Baselines: Neutral, pushed to background (z-order 2)
        'Raw': ('#7F7F7F', 'X', 2),          # Gray
        'Enhanced Raw': ('#000000', 's', 2), # Black
        
        # Autoencoders: Cool colors (Blues)
        'AE - 1/16': ('#6BAED6', 'o', 3),    # Light Blue
        'AE - 1/32': ('#08519C', '^', 3),    # Dark Blue
        
        # LWM Models: Warm colors (Oranges/Reds/Purples)
        'LWM - CLS': ('#FCAE91', 'v', 4),                  # Light Red
        'LWM - CLS (Partial Fine-Tuning)': ('#FB6A4A', 'D', 4), # Medium Red
        'LWM - CLS (Full Fine-Tuning)': ('#CB181D', '*', 5),  # Dark Red (Thicker)
        'LWM - Channel Embeddings': ('#6A51A3', 'p', 5)       # Deep Purple (Thicker)
    }

base_models = [
    "Raw",
    "AE - 1/16", "AE - 1/32", 
    "LWM - CLS", "LWM - Channel Embeddings"
            ]

fine_tune_models = [
    "Raw", "LWM - CLS",
    "LWM - CLS (Partial Fine-Tuning)", "LWM - CLS (Full Fine-Tuning)",
]

all_models = [
    "Raw",
    "AE - 1/16", "AE - 1/32", 
    "LWM - CLS", 
    "LWM - CLS (Partial Fine-Tuning)", "LWM - CLS (Full Fine-Tuning)",
    "LWM - Channel Embeddings"
]

los_nlos_train_ratios = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0]
train_ratio = [.1, .2, .3, .4, .5, .6, .7, .8, .9, 1.]

selected_snr = [-5, 0, 5, 10, 15, 20, 25]

city = 'city_18_denver'

In [ ]:
plot_coverage_map(city)

In [ ]:
from thesis.data_classes import ModelConfig
from thesis.downstream_models import build_model_from_config
from torchinfo import summary
import torch
import torch.nn as nn

class EncoderWrapper(nn.Module):
    def __init__(self, autoencoder_model):
        super().__init__()
        # "Borrow" the exact layers from your instantiated model
        self.input_norm = autoencoder_model.input_norm
        self.encoder = autoencoder_model.encoder
        self.fc_enc = autoencoder_model.fc_enc

    def forward(self, x):
        # Replicate the exact mathematical slice you want to profile
        x_real = torch.stack([x.real, x.imag], dim=1).float() 
        x_norm = self.input_norm(x_real)
        feat = self.encoder(x_norm)
        z = self.fc_enc(feat)
        return z

experiments = ["raw_model", 
               "ae_1_32", "ae_1_16",
               "lwm_cls", "lwm_cls_partial_ft",
               "lwm_cls_full_ft", "lwm_channel_emb"]

experiments_configs = []
for experiment in experiments:
    config = ModelConfig.load_json(f"../experiments/los_nlos_classification/{experiment}.json")
    model = build_model_from_config(config)

    encoder = model.encoder
    print(f"\n{experiment}")
    task_head = model.task_head

    if "ae" in experiment:
        encoder_only_model = EncoderWrapper(encoder)
        dummy_complex_input = torch.randn(1, 32, 32, dtype=torch.complex64)
        stats = summary(encoder_only_model, input_data=dummy_complex_input, verbose=0)
        print("Encoder:", "MFLOPs", stats.total_mult_adds / 1e6, "Params_M", stats.total_params / 1e6)
    if "lwm" in experiment:
        stats = summary(encoder, input_size=(1, 65, 32), verbose=0)
        print("Encoder:", "MFLOPs", stats.total_mult_adds / 1e6, "Params_M", stats.total_params / 1e6)

    stats = summary(task_head, input_size=(1, config.input_size), verbose=0)
    print("Task_Head:", "KFLOPs", stats.total_mult_adds / 1e3, "Params_L", stats.total_params / 1e3)



In [ ]:
plot_data_efficiency_results(f'../results/los_nlos_classification/{city}/data_efficiency_results.csv',
                             base_models,
                             los_nlos_train_ratios,
                             style_dict)

In [ ]:
plot_data_efficiency_results(f'../results/los_nlos_classification/{city}/data_efficiency_results.csv',
                             fine_tune_models,
                             los_nlos_train_ratios,
                             style_dict)

In [ ]:
plot_snr_robustness(f'../results/los_nlos_classification/{city}/snr_results.csv',
                    base_models,
                    1.0,
                    selected_snr,
                    style_dict)

In [ ]:
plot_snr_robustness(f'../results/los_nlos_classification/{city}/snr_results.csv',
                    fine_tune_models,
                    0.5,
                    selected_snr,
                    style_dict)

In [ ]:
plot_tsne_grid(f'../results/los_nlos_classification/{city}/user_map_los_nlos.csv',
               f'../results/los_nlos_classification/{city}/tsne_comparison_data.csv',
               all_models,
               max_cols=4)

In [ ]:
show_resources_table(f'../results/los_nlos_classification/{city}/resources_results.csv',
                     f'../results/los_nlos_classification/{city}/data_efficiency_results.csv')

In [ ]:
from thesis.data_classes import ModelConfig
from thesis.downstream_models import build_model_from_config
from torchinfo import summary
import torch
import torch.nn as nn

class EncoderWrapper(nn.Module):
    def __init__(self, autoencoder_model):
        super().__init__()
        # "Borrow" the exact layers from your instantiated model
        self.input_norm = autoencoder_model.input_norm
        self.encoder = autoencoder_model.encoder
        self.fc_enc = autoencoder_model.fc_enc

    def forward(self, x):
        # Replicate the exact mathematical slice you want to profile
        x_real = torch.stack([x.real, x.imag], dim=1).float() 
        x_norm = self.input_norm(x_real)
        feat = self.encoder(x_norm)
        z = self.fc_enc(feat)
        return z

experiments = ["raw_model", 
               "ae_1_32", "ae_1_16",
               "lwm_cls", "lwm_cls_partial_ft",
               "lwm_cls_full_ft", "lwm_channel_emb"]

experiments_configs = []
for experiment in experiments:
    config = ModelConfig.load_json(f"../experiments/beam_selection/{experiment}.json")
    model = build_model_from_config(config)

    encoder = model.encoder
    print(f"\n{experiment}")
    task_head = model.task_head

    if "ae" in experiment:
        encoder_only_model = EncoderWrapper(encoder)
        dummy_complex_input = torch.randn(1, 32, 32, dtype=torch.complex64)
        stats = summary(encoder_only_model, input_data=dummy_complex_input, verbose=0)
        print("Encoder:", "MFLOPs", stats.total_mult_adds / 1e6, "Params_M", stats.total_params / 1e6)
    if "lwm" in experiment:
        stats = summary(encoder, input_size=(1, 65, 32), verbose=0)
        print("Encoder:", "MFLOPs", round(stats.total_mult_adds / 1e6), "Params_M", round(stats.total_params / 1e6))

    stats = summary(task_head, input_size=(1, config.input_size), verbose=0)
    print("Task_Head:", "KFLOPs", round(stats.total_mult_adds / 1e3), "Params_K", round(stats.total_params / 1e3))



In [ ]:
base_models = [
    "Raw",
    "AE - 1/16", "AE - 1/32", 
    "LWM - CLS",
    "LWM - Channel Embeddings"
            ]
plot_beam_vs_data_heatmaps(f'../results/beam_selection/{city}/data_efficiency_results.csv',
                           base_models,
                           max_cols=5)

In [ ]:
plot_beam_vs_data_heatmaps(f'../results/beam_selection/{city}/data_efficiency_results.csv',
                           fine_tune_models,
                           max_cols=5)

In [ ]:
plot_beam_vs_snr_heatmaps(f'../results/beam_selection/{city}/snr_results.csv',
                          base_models,
                          1.0,
                          selected_snr,
                          max_cols=5)

In [ ]:
plot_beam_vs_snr_heatmaps(f'../results/beam_selection/{city}/snr_results.csv',
                          fine_tune_models,
                          1.0,
                          selected_snr,
                          max_cols=4)

In [ ]:
plot_tsne_grid(f'../results/beam_selection/{city}/user_map_data_16_beams.csv',
               f'../results/beam_selection/{city}/tsne_comparison_data.csv',
               all_models,
               16,
               max_cols=4)

In [ ]:
%load_ext autoreload
from thesis.metrics import *

show_resources_table(f'../results/beam_selection/{city}/resources_results.csv',
                     f'../results/beam_selection/{city}/data_efficiency_results.csv',
                     16)

In [ ]:
from thesis.data_classes import ModelConfig
from thesis.downstream_models import build_model_from_config
from torchinfo import summary
import torch
import torch.nn as nn

class EncoderWrapper(nn.Module):
    def __init__(self, autoencoder_model):
        super().__init__()
        # "Borrow" the exact layers from your instantiated model
        self.input_norm = autoencoder_model.input_norm
        self.encoder = autoencoder_model.encoder
        self.fc_enc = autoencoder_model.fc_enc

    def forward(self, x):
        # Replicate the exact mathematical slice you want to profile
        x_real = torch.stack([x.real, x.imag], dim=1).float() 
        x_norm = self.input_norm(x_real)
        feat = self.encoder(x_norm)
        z = self.fc_enc(feat)
        return z

experiments = ["raw_model", 
               "ae_1_32", "ae_1_16",
               "lwm_cls", "lwm_cls_partial_ft",
               "lwm_cls_full_ft", "lwm_channel_emb"]

experiments_configs = []
for experiment in experiments:
    config = ModelConfig.load_json(f"../experiments/power_allocation/{experiment}.json")
    config.input_size = config.input_size * 16
    model = build_model_from_config(config)
    
    print(f"\n{experiment}")
    task_head = model.task_head

    stats = summary(task_head, input_size=(1, 16, config.input_size), verbose=0)
    print("Task_Head:", "KFLOPs", stats.total_mult_adds / 1e3, "Params_L", stats.total_params / 1e3)



In [ ]:
show_resources_table(f'../results/power_allocation/{city}/resources_results.csv',
                     f'../results/power_allocation/{city}/data_efficiency_results.csv',
                     16)

In [ ]:
plot_data_efficiency_subplots(f'../results/power_allocation/{city}/aggregated_sum_rates.csv',
                           base_models,
                           train_ratio,
                           5,
                           style_dict,
                           max_cols=4)

In [ ]:
plot_noise_robustness_subplots(f'../results/power_allocation/{city}/aggregated_sum_rates.csv',
                               base_models,
                               selected_snr,
                               1.0,
                               style_dict,
                               max_cols=4)

In [ ]:
import pandas as pd
import glob
import os

# 1. Define the folder containing your CSV files
# Change this to the path where your CSV files are located
folder_path = '../results/power_allocation/city_18_denver/snr_results' 
file_pattern = os.path.join(folder_path, '*.csv')

# Find all CSV files in that folder
all_files = glob.glob(file_pattern)

if not all_files:
    print("No CSV files found in the specified directory.")
else:
    print(f"Found {len(all_files)} files. Processing...")

    # 2. Read all CSV files and combine them into a single DataFrame
    df_list = []
    for file in all_files:
        try:
            # Read the CSV; you can add 'sep' parameter if it's not comma-separated
            df = pd.read_csv(file, keep_default_na=False)
            df_list.append(df)
        except Exception as e:
            print(f"Error reading {file}: {e}")

    # Concatenate all dataframes into one large dataframe
    combined_df = pd.concat(df_list, ignore_index=True)

    # 3. Define the columns to group by
    grouping_columns = ['Num_Users', 'Train_Ratio', 'SNR_dB', 'Model_Name']

    # 4. Group by the specified columns and calculate the mean of 'Sum_Rate'
    # .reset_index() turns the grouped columns back into regular columns
    averaged_df = combined_df.groupby(grouping_columns, dropna=False)['Sum_Rate'].mean().reset_index()

    # Rename the column to reflect that it's an average (optional but recommended)
    averaged_df.rename(columns={'Sum_Rate': 'Avg_Sum_Rate'}, inplace=True)

    # 5. Save the final aggregated data to a new CSV file
    output_filename = 'aggregated_sum_rates.csv'
    averaged_df.to_csv(output_filename, index=False)
    
    print(f"Success! The averaged data has been saved to {output_filename}.")

In [ ]:
plot_tsne_grid_models_only(f'../results/power_allocation/{city}/tsne_comparison_data.csv',
                           base_models,
                           max_cols=5)